In [ ]:
!git clone https://github.com/HaiAu2501/EL4TF

Cloning into 'EL4TF'...
remote: Enumerating objects: 1768, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 1768 (delta 59), reused 64 (delta 56), pack-reused 1694 (from 3)
Receiving objects: 100% (1768/1768), 29.13 MiB | 17.33 MiB/s, done.
Resolving deltas: 100% (921/921), done.


In [ ]:
%cd EL4TF

/content/EL4TF


### training with synthetic data

In [ ]:
!pip uninstall lightgbm -y
!pip install lightgbm[gpu]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 49.1 MB/s eta 0:00:00


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

import numpy as np
from loaders._gen_binary import generate_data
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import balanced_accuracy_score, make_scorer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
_ = generate_data(verbose=True)


Generated dataset with 1995 samples.
Shapes: [(1596, 25), (0, 25), (399, 25)]
Label distribution: Counter({np.int64(1): 817, np.int64(0): 779})
[WARNING] If you use a tree-based model, consider setting use_scaler=False.


In [ ]:
bacc = []

for seed in range(30):
    pack = generate_data(seed=seed, use_scaler=False)
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    tscv = TimeSeriesSplit(n_splits=3)

    param_grid = {
        "num_leaves": [50, 100],
        "max_depth": [5, 7],
        "n_estimators": [150, 300],
        "learning_rate": [0.05, 0.1],
        # "min_child_samples": [15, 30],
        # "subsample": [0.8],          # bagging_fraction
        # "colsample_bytree": [0.8],   # feature_fraction
    }

    # Dùng balanced accuracy làm scorer chính
    scorer = make_scorer(balanced_accuracy_score)

    # Với bài toán mất cân bằng, có thể cân nhắc scale_pos_weight (nếu nhị phân)
    # nhưng để đơn giản mình không set ở đây; LGBM sẽ tự xử lý multiclass nếu có.

    base_estimator = LGBMClassifier(
        random_state=42,         # cố định model để so sánh giữa các cấu hình
        objective=None,          # để LightGBM tự suy luận binary/multiclass
        n_jobs=-1,
        verbosity=-1
    )

    search = GridSearchCV(
        estimator=base_estimator,
        param_grid=param_grid,
        cv=tscv,
        scoring=scorer,
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train, y_train)

    y_preds = search.predict(X_test)
    score = balanced_accuracy_score(y_test, y_preds)
    bacc.append(score)

    print(f"Seed {seed} - Test Balanced Accuracy: {score:.4f}")
    # Nếu muốn xem cấu hình tốt nhất cho seed này:
    # print("Best params:", search.bestparams)

print(f"Mean Test Balanced Accuracy: {np.mean(bacc)}")
print(f"Std Test Balanced Accuracy: {np.std(bacc)}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Seed 0 - Test Balanced Accuracy: 0.6741


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Seed 1 - Test Balanced Accuracy: 0.6770


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

Seed 26 - Test Balanced Accuracy: 0.6468


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Seed 27 - Test Balanced Accuracy: 0.6589


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Seed 28 - Test Balanced Accuracy: 0.6646
